# Can we predict next-month churn?
Target: `churn_next_month` on monthly account snapshots (`data.csv`, dictionary in `columns.csv`).

**Plan:** inspect data, check for leakage, build features available at snapshot time, evaluate with splits that respect accounts and time, and compare against a no-skill baseline.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay
df = pd.read_csv('data.csv')
print(df.shape, 'accounts:', df.account_ref.nunique(), '| missing:', int(df.isna().sum().sum()))
print('churn rate:', round(df.churn_next_month.mean(),4), '| months', df.snapshot_month.min(), '-', df.snapshot_month.max())
df.head()

## 1. Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from a CRM export of 2026-01-05*, so it contains information from after the snapshot. Compare it with `tenure_months` (months active up to the snapshot).

In [ ]:
gap = df.lifetime_months - df.tenure_months
print(pd.crosstab(gap.clip(upper=5), df.churn_next_month).rename_axis(index='lifetime - tenure (capped at 5)'))

`lifetime_months == tenure_months` **exactly when** the account churns, and never otherwise. It encodes the answer and would not be known for a live account at prediction time. **It is excluded from every model below.** The next cell shows the inflated score once, for contrast.

In [ ]:
y = df.churn_next_month
groups = df.account_ref
gkf = GroupKFold(n_splits=5)
p = cross_val_predict(HistGradientBoostingClassifier(random_state=0), df[['tenure_months','lifetime_months']], y, cv=gkf, groups=groups, method='predict_proba')[:,1]
print('AUC using lifetime_months (LEAKY, do not trust):', round(roc_auc_score(y,p),3))

## 2. Exploration (legitimate features only)

In [ ]:
df['month'] = pd.PeriodIndex(df.snapshot_month, freq='M')
df['fee_per_seat'] = df.monthly_fee / df.seats
num = ['seats','monthly_fee','fee_per_seat','tenure_months','usage_hours','support_tickets']
print(df.groupby('plan')[['seats','monthly_fee','fee_per_seat']].median())
fig, ax = plt.subplots(2,3, figsize=(14,7))
for a,c in zip(ax.ravel(), num):
    q = pd.qcut(df[c], 8, duplicates='drop') if df[c].nunique()>8 else df[c]
    df.groupby(q, observed=True).churn_next_month.mean().plot(marker='o', ax=a)
    a.set_title(f'churn rate by {c}'); a.set_ylabel('rate'); a.tick_params(axis='x', rotation=45); a.set_xlabel('')
plt.tight_layout(); plt.show()
df.groupby('month').churn_next_month.agg(['mean','size']).T.round(3)

Rows are repeated snapshots of the same 3,000 accounts, so a random row split would put one account in both train and test. I use **GroupKFold by account**, plus a **time-based holdout** (train on earlier months, test on the latest).

## 3. Models

In [ ]:
feats = ['plan','seats','monthly_fee','fee_per_seat','tenure_months','usage_hours','support_tickets']
X = pd.get_dummies(df[feats], columns=['plan'], dtype=float)
models = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'grad. boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}
def report(name, y_true, p): return dict(model=name, AUC=roc_auc_score(y_true,p), AP=average_precision_score(y_true,p))
rows = [report('base rate (no skill)', y, np.full(len(y), y.mean()))]
oof = {}
for n,m in models.items():
    oof[n] = cross_val_predict(m, X, y, cv=gkf, groups=groups, method='predict_proba')[:,1]
    rows.append(report(n, y, oof[n]))
print('Grouped 5-fold CV')
pd.DataFrame(rows).set_index('model').round(3)

In [ ]:
# time-based holdout: last 4 snapshot months as test, restricted to accounts never seen in training
cut = sorted(df.month.unique())[-4]
test_accts = set(df.account_ref[df.month >= cut]) - set(df.account_ref[df.month < cut])
tr = (df.month < cut).values
te = ((df.month >= cut) & df.account_ref.isin(test_accts)).values
rows = [report('base rate (no skill)', y[te], np.full(te.sum(), y[tr].mean()))]
for n,m in models.items():
    m.fit(X[tr], y[tr]); rows.append(report(n, y[te], m.predict_proba(X[te])[:,1]))
print('train rows', tr.sum(), '| test rows', te.sum(), '| test churn rate', round(y[te].mean(),3))
pd.DataFrame(rows).set_index('model').round(3)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4.5))
for n,p in oof.items(): RocCurveDisplay.from_predictions(y, p, name=n, ax=ax[0])
ax[0].plot([0,1],[0,1],'k:'); ax[0].set_title('ROC (grouped CV)')
d = pd.DataFrame({'y':y,'p':oof['grad. boosting']}); d['decile'] = pd.qcut(d.p,10,labels=False)
d.groupby('decile').y.mean().plot.bar(ax=ax[1]); ax[1].axhline(y.mean(), color='k', ls=':')
ax[1].set_title('Actual churn rate by predicted-risk decile (grouped CV)'); plt.tight_layout(); plt.show()

In [ ]:
from sklearn.inspection import permutation_importance
m = models['grad. boosting'].fit(X, y)
pi = permutation_importance(m, X, y, scoring='roc_auc', n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, X.columns).sort_values().plot.barh(title='Permutation importance (AUC drop, in-sample)'); plt.show()

## 4. Conclusion
(Numbers from the run that produced this notebook; re-run to refresh.)

- **Leakage:** `lifetime_months` gives AUC 1.0 because it equals `tenure_months` only for churners. It is excluded; any model using it is not a real forecast.
- **Honest performance:** grouped 5-fold CV AUC is about 0.67-0.68 (average precision about 0.28-0.30 vs a 0.154 base rate). On a time-based holdout of unseen accounts, AUC is about 0.71 (AP about 0.24 vs 0.097). That is real but modest signal: better than chance, far from reliable per-account prediction.
- **Best use:** ranking accounts by risk (e.g. targeting the top deciles for outreach), not confident individual calls. Logistic regression and gradient boosting perform about the same, so the signal is mostly simple.
- **Caveats:** churn rate varies a lot by calendar month (about 23% in early 2024 falling to 3% and rising again), so the holdout is based on few months and may not transfer. Only one snapshot per account-month and no history features (usage trend, ticket trend) are used; adding lagged features is the obvious next step.